# 🛫🛬🗼🗽😎 <span style="color: white; background-color: Purple"><b> Relatório de Férias </b></span></p>

🧩 <span style="color: MediumSlateBlue"><b> 1- Carregamento das bases essenciais </b></span></p>
- O script importa quatro tabelas principais:
    - COLABORADORES.xlsx — informações completas dos colaboradores  
    - UNIDADES.xlsx — nomes e códigos das unidades/resumo empresarial  
    - EMAIL.xlsx — base opcional com endereços de e‑mail  
    - PROCESSOS.xlsx — arquivo de auditoria das execuções
- Cada base é convertida em DataFrames pandas de forma robusta, com validações de integridade

📅 <span style="color: MediumSlateBlue"><b> 2- Cálculo automático do período de férias </b></span></p>
- O sistema calcula dinamicamente o próximo período mensal de férias utilizando dateutil.relativedelta
- O pipeline gera:
    - Data de início → primeiro dia do mês seguinte  
    - Data de fim → último dia do mês seguinte  
    - Strings personalizadas para o nome do arquivo:  YYYYMMDD  
    - mês abreviado em PT‑BR (JAN, FEV, MAR etc.)  
    - ano
- Esse cálculo garante que o relatório seja mensal e automatizado, sem ajustes manuais

🧮 <span style="color: MediumSlateBlue"><b> 3- Filtragem dos colaboradores com férias no período </b></span></p>
- Do arquivo de colaboradores, o sistema filtra automaticamente:
    - Apenas ativos  
    - Que possuam data_inicio_ferias dentro do período calculado
- E remove colaboradores sem data ou com registros inconsistentes

🔗 <span style="color: MediumSlateBlue"><b> 4- Enriquecimento da base (Joins) </b></span></p>
- Join com unidades
- Via cod_empresa, trazendo:
    - resumo da empresa  
    - unidade  
    - agrupamento organizacional
- Join com e‑mails
    - e‑mail preferencial  
    - aumentando completude dos contatos
- Caso o arquivo EMAIL.xlsx não exista, o script continua a execução e insere colunas vazias sem quebrar

🧼 <span style="color: MediumSlateBlue"><b> 5- Padronização e renomeação das colunas </b></span></p>
- A base final passa a ter exatamente as colunas:
    - Matrícula  
    - Nome  
    - Cargo  
    - Centro de custo  
    - Unidades  
    - Início das férias  
    - Fim das férias  
    - E‑mail
- Com formatação estruturada e organizada para facilitar leitura e distribuição

📁 <span style="color: MediumSlateBlue"><b> 6- Exportação do relatório final </b></span></p>
- O arquivo é salvo em: 10 – Relatórios / 10.2 – Relatório de Férias / 2026
- Com o nome estruturado: RELATÓRIO DE FERIAS {YYYYMMDD} {MÊS_ABREV} {ANO}.xlsx

🧾 <span style="color: MediumSlateBlue"><b> 7- Registro completo da execução no PROCESSOS.xlsx </b></span></p>
- O pipeline registra automaticamente:

    🟦 ETAPA 0 — Início  
    🟩 ETAPA 1 — Importação  
    🟨 ETAPA 2 — Preparação de variáveis  
    🟧 ETAPA 3 — Processamento  
    🟥 ETAPA 4 — Exportação
- Cada etapa com:
    - ID do processo (14)  
    - timestamp exato  
    - número da etapa
- Esse log permite auditoria completa e histórico de execuções

📊 <span style="color: MediumSlateBlue"><b> 8- Relatório final na tela </b></span></p>
- Quantidade de colaboradores no relatório  
- Nome exato do arquivo gerado  
- Caminho final  
- Tempo total da execução

In [1]:
# IMPORTANDO AS BIBLIOTECAS
import pandas as pd
import sys
import logging
from pathlib import Path
from datetime import datetime, date
from dateutil.relativedelta import relativedelta
from openpyxl import load_workbook

# CONFIGURAÇÃO DE LOGGING
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

# CONFIGURAÇÕES GERAIS E CAMINHOS
BASE_DIR = Path(r'X:\Gestão de Pessoas\Analytics')

PATH_PROCESSOS     = BASE_DIR / '03 - Bases' / '1. BASES TRATADAS' / 'PROCESSOS.xlsx'
PATH_COLABORADORES = BASE_DIR / '03 - Bases' / '1. BASES TRATADAS' / 'COLABORADORES.xlsx'
PATH_UNIDADES      = BASE_DIR / '03 - Bases' / '1. BASES TRATADAS' / 'UNIDADES.xlsx'
PATH_EMAIL         = BASE_DIR / '03 - Bases' / '1. BASES TRATADAS' / 'EMAIL.xlsx'
PATH_OUTPUT_DIR    = BASE_DIR / '10 - Relatórios' / '10.2 - Relatório de férias/2026'

PROCESS_ID = 14


def registrar_log_excel(etapa: int, process_id: int = PROCESS_ID, filepath: Path = PATH_PROCESSOS) -> None:
    """Registra a etapa do processo no arquivo Excel de controle."""
    try:
        wb = load_workbook(filepath)
        ws = wb['REGISTROS']
        nova_linha = [process_id, datetime.now(), etapa]
        ws.append(nova_linha)
        wb.save(filepath)
        wb.close()
        logger.info(f"Log de etapa {etapa} registrado com sucesso.")
    except PermissionError:
        logger.error(f"ERRO: O arquivo {filepath.name} está aberto. Log não gravado.")
    except Exception as e:
        logger.error(f"Erro ao registrar log: {e}")


def get_periodo_ferias() -> tuple[date, date, str, str, str]:
    """Calcula as datas de corte e strings para nomeação do arquivo."""
    hoje = datetime.now()

    inicio_ferias = (hoje + relativedelta(months=1)).replace(day=1).date()
    fim_ferias = (hoje + relativedelta(months=2)).replace(day=1) - relativedelta(days=1)
    fim_ferias = fim_ferias.date()

    date_name = hoje.strftime('%Y%m%d')
    meses_pt = ["JAN", "FEV", "MAR", "ABR", "MAI", "JUN", "JUL", "AGO", "SET", "OUT", "NOV", "DEZ"]
    mes_ref_custom = meses_pt[(hoje + relativedelta(months=1)).month - 1]
    ano_ref = str((hoje + relativedelta(months=1)).year)

    return inicio_ferias, fim_ferias, date_name, mes_ref_custom, ano_ref


# FUNÇÃO PRINCIPAL
def main():
    start_time = datetime.now()
    logger.info("Iniciando processo de Relatório de Férias (Pandas Version)...")

    # --- ETAPA 0: Início ---
    registrar_log_excel(etapa=0)

    try:
        # --- ETAPA 1: Importação ---
        logger.info("Carregando bases de dados...")

        colaboradores = pd.read_excel(PATH_COLABORADORES)
        unidades = pd.read_excel(PATH_UNIDADES)

        # Carrega a base de e-mails (aba CONTATO)
        # Em caso de erro no caminho ou colunas, loga aviso e segue sem e-mail
        try:
            emails = pd.read_excel(PATH_EMAIL, sheet_name='CONTATO')
            # Normaliza os nomes das colunas para evitar falhas por capitalização
            emails.columns = emails.columns.str.strip().str.lower()
            # Normaliza os valores da coluna 'nome' para UPPERCASE sem espaços extras
            emails['nome'] = emails['nome'].astype(str).str.strip().str.upper()
            emails = emails[['nome', 'email']].drop_duplicates(subset=['nome'])
        except Exception as e:
            logger.warning(f"Não foi possível carregar EMAIL.xlsx: {e}. Coluna de e-mail ficará em branco.")
            emails = pd.DataFrame(columns=['nome', 'email'])

        registrar_log_excel(etapa=1)

        # --- ETAPA 2: Preparação de Variáveis ---
        dt_inicio, dt_fim, str_data, str_mes, str_ano = get_periodo_ferias()

        nome_arquivo = f"RELATÓRIO DE FERIAS {str_data} {str_mes} {str_ano}.xlsx"
        path_salvamento = PATH_OUTPUT_DIR / nome_arquivo

        logger.info(f"Período de análise: {dt_inicio} a {dt_fim}")
        registrar_log_excel(etapa=2)

        # --- ETAPA 3: Processamento (Pandas) ---
        logger.info("Processando dados com Pandas...")

        # 1. Certificar que as colunas de data estão no formato datetime para comparação
        colaboradores['data_inicio_ferias'] = pd.to_datetime(colaboradores['data_inicio_ferias']).dt.date
        colaboradores['data_fim_ferias'] = pd.to_datetime(colaboradores['data_fim_ferias']).dt.date

        # 2. Filtragem: Ativos e dentro do período
        df_filtrado = colaboradores[
            (colaboradores['data_inicio_ferias'] >= dt_inicio) &
            (colaboradores['data_inicio_ferias'] <= dt_fim) &
            (colaboradores['descricao_rescisao'] == "ATIVO")
        ].copy()

        # 3. Cruzamento (Merge) com a base de Unidades
        # Left Join para garantir que não perdemos colaboradores se a unidade não for encontrada
        base_final = pd.merge(
            df_filtrado,
            unidades[['cod_empresa', 'resumo_empresa']],
            on='cod_empresa',
            how='inner'
        )

        # 4. Seleção e Renomeação de colunas (equivalente ao SELECT AS do SQL)
        colunas_map = {
            'registro': 'Matrícula',
            'nome': 'Nome',
            'cargo_completo': 'Cargo',
            'centro_custo': 'Centro de custo',
            'resumo_empresa': 'Unidades',
            'data_inicio_ferias': 'Início das férias',
            'data_fim_ferias': 'Fim das férias'
        }

        # Filtra apenas as colunas mapeadas e renomeia
        base_final = base_final[list(colunas_map.keys())].rename(columns=colunas_map)

        # 5. Cruzamento com a base de e-mails pelo nome do colaborador
        # Normaliza 'Nome' no relatório para UPPERCASE (mesmo padrão do EMAIL.xlsx)
        base_final['_nome_key'] = base_final['Nome'].astype(str).str.strip().str.upper()

        base_final = pd.merge(
            base_final,
            emails.rename(columns={'nome': '_nome_key'}),
            on='_nome_key',
            how='left'
        )

        # Remove a coluna auxiliar de chave e garante a coluna 'E-mail' no resultado
        base_final = base_final.drop(columns=['_nome_key'])
        base_final['email'] = base_final['email'].fillna('')
        base_final = base_final.rename(columns={'email': 'E-mail'})

        registrar_log_excel(etapa=3)

        # --- ETAPA 4: Exportação ---
        logger.info(f"Salvando arquivo em: {path_salvamento}")
        base_final.to_excel(path_salvamento, index=False)

        registrar_log_excel(etapa=4)

        # Resumo Final
        duration = datetime.now() - start_time
        print('-' * 80)
        print(f'   ✅ Processo finalizado com SUCESSO')
        print(f'   📂 Arquivo gerado: {nome_arquivo}')
        print(f'   📧 Colaboradores com e-mail: {(base_final["E-mail"] != "").sum()} / {len(base_final)}')
        print(f'   ⏱️ Tempo de execução: {duration}')
        print('-' * 80)

    except Exception as e:
        logger.critical(f"FALHA CRÍTICA NO PROCESSO: {e}")
        sys.exit(1)


if __name__ == "__main__":
    main()

09:47:11 - INFO - Iniciando processo de Relatório de Férias (Pandas Version)...
09:47:11 - INFO - Log de etapa 0 registrado com sucesso.
09:47:11 - INFO - Carregando bases de dados...
09:47:17 - INFO - Log de etapa 1 registrado com sucesso.
09:47:17 - INFO - Período de análise: 2026-07-01 a 2026-07-31
09:47:18 - INFO - Log de etapa 2 registrado com sucesso.
09:47:18 - INFO - Processando dados com Pandas...
09:47:18 - INFO - Log de etapa 3 registrado com sucesso.
09:47:18 - INFO - Salvando arquivo em: X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.2 - Relatório de férias\2026\RELATÓRIO DE FERIAS 20260622 JUL 2026.xlsx
09:47:19 - INFO - Log de etapa 4 registrado com sucesso.


--------------------------------------------------------------------------------
   ✅ Processo finalizado com SUCESSO
   📂 Arquivo gerado: RELATÓRIO DE FERIAS 20260622 JUL 2026.xlsx
   📧 Colaboradores com e-mail: 100 / 102
   ⏱️ Tempo de execução: 0:00:07.581682
--------------------------------------------------------------------------------
